In [1]:
# # ============================================================
# # CELL 1 — Install tambahan (TIDAK install ulang torch!)
# # ============================================================
!pip install facenet-pytorch --no-deps -q
!pip install ultralytics -q
!pip install pandas scikit-learn tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.7 MB/s eta 0:00:00


In [2]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import os
import gc
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import timm

from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from facenet_pytorch import MTCNN

print("✓ Semua import berhasil")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ Semua import berhasil


In [3]:

# ============================================================
# CELL 3 — Device
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")



Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [4]:

# ============================================================
# CELL 4 — Load detection models
# ============================================================
yolo_model   = YOLO("yolov8n.pt")
retina_model = MTCNN(keep_all=True, device=device)
print("✓ YOLO & MTCNN loaded")




✓ YOLO & MTCNN loaded


In [5]:

# ============================================================
# CELL 5 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [6]:

# ============================================================
# CELL 6 — Path config (ubah sesuai struktur folder kamu)
# ============================================================
TRAIN_PATH   = "/content/drive/MyDrive/DATASET/data-analytics-competition-dac-find-it-2026/train"
TEST_PATH    = "/content/drive/MyDrive/DATASET/data-analytics-competition-dac-find-it-2026/test"
FACES_DIR    = "/content/drive/MyDrive/faces"
TRAIN_CSV    = f"{FACES_DIR}/train_df_processed.csv"
TEST_CSV     = f"{FACES_DIR}/test_df_processed.csv"
TRAIN_FACES  = f"{FACES_DIR}/train"
TEST_FACES   = f"{FACES_DIR}/test"


In [7]:

# ============================================================
# CELL 7 — Dataset loaders
# ============================================================
def load_train_dataset(train_path, limit=None):
    train_path = Path(train_path)
    class_dirs = sorted([d for d in train_path.iterdir() if d.is_dir()])
    label_to_id = {cls.name: idx for idx, cls in enumerate(class_dirs)}

    image_paths, labels = [], []
    for cls in class_dirs:
        files = (
            list(cls.glob("*.jpg")) +
            list(cls.glob("*.png")) +
            list(cls.glob("*.jpeg"))
        )
        if limit:
            files = files[:limit]
        for p in files:
            image_paths.append(str(p))
            labels.append(label_to_id[cls.name])

    return image_paths, labels, label_to_id


def load_test_dataset(test_path):
    test_path = Path(test_path)
    files = (
        list(test_path.glob("*.jpg")) +
        list(test_path.glob("*.png")) +
        list(test_path.glob("*.jpeg"))
    )
    return [str(p) for p in files], [p.stem for p in files]


def build_train_df(image_paths, labels):
    return pd.DataFrame({"path": image_paths, "label": labels})


def build_test_df(image_paths, image_ids):
    return pd.DataFrame({"path": image_paths, "id": image_ids})


# load
train_paths, train_labels, label_map = load_train_dataset(TRAIN_PATH)
test_paths,  test_ids               = load_test_dataset(TEST_PATH)

train_df = build_train_df(train_paths, train_labels)
test_df  = build_test_df(test_paths,  test_ids)

print(f"label_map : {label_map}")
print(f"Train raw : {len(train_df)} | Test raw: {len(test_df)}")
print("Distribusi:\n", train_df["label"].value_counts().sort_index())




label_map : {'fake_mannequin': 0, 'fake_mask': 1, 'fake_printed': 2, 'fake_screen': 3, 'fake_unknown': 4, 'realperson': 5}
Train raw : 1438 | Test raw: 404
Distribusi:
 label
0    196
1    251
2     96
3    188
4    298
5    409
Name: count, dtype: int64


In [8]:

# ============================================================
# CELL 8 — Face detection helper
# ============================================================
def detect_face_ensemble(
    image_path,
    yolo_model,
    retina_model,
    conf=0.3,
    pad_tight=0.05,
    pad_loose=0.2,
    fallback=True,
):
    img = cv2.imread(image_path)
    if img is None:
        return None, None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    candidates = []

    # YOLO — pakai PIL Image (hindari konflik numpy internal)
    try:
        img_pil = Image.fromarray(img)
        yolo_results = yolo_model(img_pil, conf=conf, verbose=False)[0]
        if len(yolo_results.boxes) > 0:
            for b in yolo_results.boxes.xyxy.cpu().numpy():
                x1, y1, x2, y2 = b
                candidates.append([x1, y1, x2, y2, (x2 - x1) * (y2 - y1)])
    except Exception:
        pass

    # MTCNN
    try:
        boxes, _ = retina_model.detect(img)
        if boxes is not None:
            for b in boxes:
                x1, y1, x2, y2 = b
                candidates.append([x1, y1, x2, y2, (x2 - x1) * (y2 - y1)])
    except Exception:
        pass

    if len(candidates) == 0:
        if fallback:
            return img, img
        return None, None

    candidates = np.array(candidates)
    x1, y1, x2, y2 = candidates[np.argmax(candidates[:, 4])][:4]
    bw, bh = x2 - x1, y2 - y1

    tx1 = int(max(0, x1 - pad_tight * bw)); ty1 = int(max(0, y1 - pad_tight * bh))
    tx2 = int(min(w,  x2 + pad_tight * bw)); ty2 = int(min(h,  y2 + pad_tight * bh))
    tight = img[ty1:ty2, tx1:tx2]

    lx1 = int(max(0, x1 - pad_loose * bw)); ly1 = int(max(0, y1 - pad_loose * bh))
    lx2 = int(min(w,  x2 + pad_loose * bw)); ly2 = int(min(h,  y2 + pad_loose * bh))
    loose = img[ly1:ly2, lx1:lx2]

    return tight, loose


def process_dataset_dual(df, save_dir, yolo_model, retina_model):
    os.makedirs(save_dir, exist_ok=True)
    tight_paths, loose_paths = [], []

    for i in tqdm(range(len(df))):
        img_path = df.iloc[i]["path"]
        tight, loose = detect_face_ensemble(img_path, yolo_model, retina_model)

        if tight is None or loose is None or tight.size == 0 or loose.size == 0:
            tight_paths.append(None)
            loose_paths.append(None)
            continue

        tp = os.path.join(save_dir, f"{i}_tight.jpg")
        lp = os.path.join(save_dir, f"{i}_loose.jpg")
        cv2.imwrite(tp, cv2.cvtColor(tight, cv2.COLOR_RGB2BGR))
        cv2.imwrite(lp, cv2.cvtColor(loose, cv2.COLOR_RGB2BGR))
        tight_paths.append(tp)
        loose_paths.append(lp)

    df = df.copy()
    df["face_tight"] = tight_paths
    df["face_loose"] = loose_paths
    return df




In [9]:

# ============================================================
# CELL 9 — Proses & simpan face crops
# Jalankan SEKALI saja, lalu comment semua blok ini
# ============================================================

# --- TRAIN ---
# os.makedirs(FACES_DIR, exist_ok=True)
# train_df = process_dataset_dual(train_df, TRAIN_FACES, yolo_model, retina_model)
# train_df.to_csv(TRAIN_CSV, index=False)
# print(f"✓ Train saved: {TRAIN_CSV}")

# # --- TEST ---
# test_df = process_dataset_dual(test_df, TEST_FACES, yolo_model, retina_model)
# test_df.to_csv(TEST_CSV, index=False)
# print(f"✓ Test saved: {TEST_CSV}")




In [10]:

# ============================================================
# CELL 10 — Load CSV (gunakan ini di sesi berikutnya)
# ============================================================
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

train_df = train_df.dropna(subset=["face_tight", "face_loose"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["face_tight", "face_loose"]).reset_index(drop=True)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print("Distribusi:\n", train_df["label"].value_counts().sort_index())



Train: 1438 | Test: 404
Distribusi:
 label
0    196
1    251
2     96
3    188
4    298
5    409
Name: count, dtype: int64


In [11]:

# ============================================================
# CELL 11 — Train / Val split
# ============================================================
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=42,
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")

Train: 1150 | Val: 288


In [12]:

# ============================================================
# CELL 12 — Transforms
# Augmentasi agresif untuk atasi data sedikit & overfitting
# ============================================================
train_tf = T.Compose([
    T.Resize((256, 256)),
    T.RandomCrop((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=20),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.RandomPerspective(distortion_scale=0.3, p=0.3),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.3, scale=(0.02, 0.2)),
])

val_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [13]:

# ============================================================
# CELL 13 — Dataset classes
# ============================================================
class FaceDataset(Dataset):
    def __init__(self, df, transform=None, is_train=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.is_train  = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        img_tight = Image.open(row["face_tight"]).convert("RGB")
        img_loose = Image.open(row["face_loose"]).convert("RGB")
        label     = int(row["label"])

        if self.is_train:
            if random.random() < 0.5:
                img_tight = TF.hflip(img_tight)
                img_loose = TF.hflip(img_loose)
            angle     = random.uniform(-15, 15)
            img_tight = TF.rotate(img_tight, angle)
            img_loose = TF.rotate(img_loose, angle)

        if self.transform:
            img_tight = self.transform(img_tight)
            img_loose = self.transform(img_loose)

        return img_tight, img_loose, label


class TestFaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        img_tight = Image.open(row["face_tight"]).convert("RGB")
        img_loose = Image.open(row["face_loose"]).convert("RGB")
        img_id    = str(row["id"])

        if self.transform:
            img_tight = self.transform(img_tight)
            img_loose = self.transform(img_loose)

        return img_tight, img_loose, img_id




In [14]:

# ============================================================
# CELL 14 — DataLoaders + WeightedRandomSampler
# ============================================================
def make_weighted_sampler(df):
    counts  = df["label"].value_counts().sort_index().values.astype(float)
    weights = 1.0 / counts
    sw      = [weights[lbl] for lbl in df["label"]]
    return WeightedRandomSampler(
        torch.DoubleTensor(sw), num_samples=len(sw), replacement=True
    )

train_ds = FaceDataset(train_df, transform=train_tf, is_train=True)
val_ds   = FaceDataset(val_df,   transform=val_tf,   is_train=False)
test_ds  = TestFaceDataset(test_df, transform=val_tf)

sampler = make_weighted_sampler(train_df)

train_loader = DataLoader(train_ds, batch_size=8, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")



Train batches: 144 | Val: 36 | Test: 51


In [15]:

# ============================================================
# CELL 15 — Model definitions
# Regularisasi kuat: dropout 0.5, BatchNorm, drop_path
# ============================================================
def _make_classifier(in_features, num_classes):
    return nn.Sequential(
        nn.Linear(in_features * 2, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )


class DualEfficientNet(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.backbone = timm.create_model(
            "efficientnet_b3", pretrained=True, num_classes=0,
            drop_rate=0.3, drop_path_rate=0.2,
        )
        self.classifier = _make_classifier(self.backbone.num_features, num_classes)

    def forward(self, x_tight, x_loose):
        return self.classifier(
            torch.cat([self.backbone(x_tight), self.backbone(x_loose)], dim=1)
        )


class DualConvNeXt(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.backbone = timm.create_model(
            "convnext_tiny", pretrained=True, num_classes=0,
            drop_rate=0.3, drop_path_rate=0.2,
        )
        self.backbone.set_grad_checkpointing(True)
        self.classifier = _make_classifier(self.backbone.num_features, num_classes)

    def forward(self, x_tight, x_loose):
        return self.classifier(
            torch.cat([self.backbone(x_tight), self.backbone(x_loose)], dim=1)
        )


class DualViT(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.backbone = timm.create_model(
            "swin_tiny_patch4_window7_224", pretrained=True, num_classes=0,
            drop_rate=0.3, drop_path_rate=0.2,
        )
        self.backbone.set_grad_checkpointing(True)
        self.classifier = _make_classifier(self.backbone.num_features, num_classes)

    def forward(self, x_tight, x_loose):
        return self.classifier(
            torch.cat([self.backbone(x_tight), self.backbone(x_loose)], dim=1)
        )




In [16]:

# ============================================================
# CELL 16 — Weighted loss (atasi class imbalance)
# ============================================================
def get_criterion(train_df, device):
    counts  = train_df["label"].value_counts().sort_index().values.astype(float)
    w       = torch.FloatTensor(1.0 / counts)
    w       = (w / w.sum() * len(counts)).to(device)
    return nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)



In [17]:

# ============================================================
# CELL 17 — Epoch loop helper
# ============================================================
def _run_epochs(
    model, train_loader, val_loader,
    optimizer, scheduler, scaler, criterion,
    device, use_amp, epochs, start_epoch,
    best_acc, save_path, model_name,
):
    for epoch in range(epochs):
        torch.cuda.empty_cache(); gc.collect()
        ep = start_epoch + epoch + 1
        print(f"  [{model_name}] Epoch {ep}/{start_epoch + epochs}")

        # ----- TRAIN -----
        model.train()
        train_loss = 0.0
        for img_tight, img_loose, labels in train_loader:
            img_tight = img_tight.to(device, non_blocking=True)
            img_loose = img_loose.to(device, non_blocking=True)
            labels    = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                out  = model(img_tight, img_loose)
                loss = criterion(out, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            del img_tight, img_loose, labels, out, loss

        # ----- VALIDATION -----
        model.eval()
        val_loss = 0.0; correct = 0; total = 0
        with torch.no_grad():
            for img_tight, img_loose, labels in val_loader:
                img_tight = img_tight.to(device, non_blocking=True)
                img_loose = img_loose.to(device, non_blocking=True)
                labels    = labels.to(device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=use_amp):
                    out  = model(img_tight, img_loose)
                    loss = criterion(out, labels)

                val_loss += loss.item()
                correct  += (out.argmax(1) == labels).sum().item()
                total    += labels.size(0)
                del img_tight, img_loose, labels, out, loss

        val_acc = correct / total
        print(f"    Train Loss: {train_loss/len(train_loader):.4f} | "
              f"Val Loss: {val_loss/len(val_loader):.4f} | "
              f"Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"    ✓ Saved (acc={best_acc:.4f})")

        scheduler.step()

    return best_acc


In [18]:

# ============================================================
# CELL 18 — Main train function (Freeze → Unfreeze)
# ============================================================
def train_model(
    model, train_loader, val_loader, device,
    save_path="best.pth", model_name="Model",
    warmup_epochs=5,    finetune_epochs=25,
    warmup_lr=1e-3,     backbone_lr=1e-5,    head_lr=1e-4,
):
    use_amp   = device.type == "cuda"
    scaler    = torch.cuda.amp.GradScaler(enabled=use_amp)
    criterion = get_criterion(train_df, device)
    best_acc  = 0.0

    # FASE 1 — freeze backbone, warmup classifier
    for p in model.backbone.parameters():
        p.requires_grad = False

    opt = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=warmup_lr, weight_decay=1e-2)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=warmup_epochs)

    print(f"\n[{model_name}] FASE 1 — Warmup classifier ({warmup_epochs} epoch)")
    best_acc = _run_epochs(model, train_loader, val_loader, opt, sch, scaler,
                           criterion, device, use_amp, warmup_epochs, 0,
                           best_acc, save_path, model_name)

    # FASE 2 — unfreeze semua, fine-tune dengan lr berbeda
    for p in model.backbone.parameters():
        p.requires_grad = True

    opt = optim.AdamW([
        {"params": model.backbone.parameters(),   "lr": backbone_lr},
        {"params": model.classifier.parameters(), "lr": head_lr},
    ], weight_decay=1e-2)
    sch = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=1)

    print(f"\n[{model_name}] FASE 2 — Fine-tune semua layer ({finetune_epochs} epoch)")
    best_acc = _run_epochs(model, train_loader, val_loader, opt, sch, scaler,
                           criterion, device, use_amp, finetune_epochs, warmup_epochs,
                           best_acc, save_path, model_name)

    print(f"\n[{model_name}] Selesai. Best Val Acc: {best_acc:.4f}")
    return model




In [19]:

# ============================================================
# CELL 19 — Train EfficientNet-B3
# ============================================================
model_eff = DualEfficientNet(num_classes=6).to(device)
model_eff = train_model(
    model_eff, train_loader, val_loader, device,
    save_path="/content/drive/MyDrive/faces/eff_best.pth",
    model_name="EfficientNet-B3",
)
del model_eff; torch.cuda.empty_cache(); gc.collect()



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

/tmp/ipykernel_2188/2153278407.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=use_amp)



[EfficientNet-B3] FASE 1 — Warmup classifier (5 epoch)
  [EfficientNet-B3] Epoch 1/5


/tmp/ipykernel_2188/1462884191.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
/tmp/ipykernel_2188/1462884191.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


    Train Loss: 1.2975 | Val Loss: 1.1471 | Val Acc: 0.7118
    ✓ Saved (acc=0.7118)
  [EfficientNet-B3] Epoch 2/5
    Train Loss: 1.0468 | Val Loss: 1.0707 | Val Acc: 0.7049
  [EfficientNet-B3] Epoch 3/5
    Train Loss: 1.0709 | Val Loss: 0.9873 | Val Acc: 0.7535
    ✓ Saved (acc=0.7535)
  [EfficientNet-B3] Epoch 4/5
    Train Loss: 0.9718 | Val Loss: 0.9732 | Val Acc: 0.7500
  [EfficientNet-B3] Epoch 5/5
    Train Loss: 0.9327 | Val Loss: 0.9639 | Val Acc: 0.7778
    ✓ Saved (acc=0.7778)

[EfficientNet-B3] FASE 2 — Fine-tune semua layer (25 epoch)
  [EfficientNet-B3] Epoch 6/30
    Train Loss: 0.9265 | Val Loss: 0.9396 | Val Acc: 0.7951
    ✓ Saved (acc=0.7951)
  [EfficientNet-B3] Epoch 7/30
    Train Loss: 0.8692 | Val Loss: 0.8882 | Val Acc: 0.8438
    ✓ Saved (acc=0.8438)
  [EfficientNet-B3] Epoch 8/30
    Train Loss: 0.8885 | Val Loss: 0.8825 | Val Acc: 0.8576
    ✓ Saved (acc=0.8576)
  [EfficientNet-B3] Epoch 9/30
    Train Loss: 0.8495 | Val Loss: 0.8609 | Val Acc: 0.8368
  [Ef

0

In [20]:

# ============================================================
# CELL 20 — Train ConvNeXt-Tiny
# ============================================================
model_cnx = DualConvNeXt(num_classes=6).to(device)
model_cnx = train_model(
    model_cnx, train_loader, val_loader, device,
    save_path="/content/drive/MyDrive/faces/cnx_best.pth",
    model_name="ConvNeXt-Tiny",
)
del model_cnx; torch.cuda.empty_cache(); gc.collect()



model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

/tmp/ipykernel_2188/2153278407.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=use_amp)



[ConvNeXt-Tiny] FASE 1 — Warmup classifier (5 epoch)
  [ConvNeXt-Tiny] Epoch 1/5


/tmp/ipykernel_2188/1462884191.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
/tmp/ipykernel_2188/1462884191.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


    Train Loss: 1.1052 | Val Loss: 0.8573 | Val Acc: 0.8472
    ✓ Saved (acc=0.8472)
  [ConvNeXt-Tiny] Epoch 2/5
    Train Loss: 0.8595 | Val Loss: 0.8161 | Val Acc: 0.8889
    ✓ Saved (acc=0.8889)
  [ConvNeXt-Tiny] Epoch 3/5
    Train Loss: 0.8498 | Val Loss: 0.7724 | Val Acc: 0.8993
    ✓ Saved (acc=0.8993)
  [ConvNeXt-Tiny] Epoch 4/5
    Train Loss: 0.8021 | Val Loss: 0.7939 | Val Acc: 0.8993
  [ConvNeXt-Tiny] Epoch 5/5
    Train Loss: 0.7673 | Val Loss: 0.7646 | Val Acc: 0.9167
    ✓ Saved (acc=0.9167)

[ConvNeXt-Tiny] FASE 2 — Fine-tune semua layer (25 epoch)
  [ConvNeXt-Tiny] Epoch 6/30
    Train Loss: 0.7617 | Val Loss: 0.7525 | Val Acc: 0.8993
  [ConvNeXt-Tiny] Epoch 7/30
    Train Loss: 0.7048 | Val Loss: 0.7016 | Val Acc: 0.9444
    ✓ Saved (acc=0.9444)
  [ConvNeXt-Tiny] Epoch 8/30
    Train Loss: 0.6584 | Val Loss: 0.6725 | Val Acc: 0.9410
  [ConvNeXt-Tiny] Epoch 9/30
    Train Loss: 0.6371 | Val Loss: 0.6851 | Val Acc: 0.9444
  [ConvNeXt-Tiny] Epoch 10/30
    Train Loss: 0.

168

In [21]:


# ============================================================
# CELL 21 — Train Swin-Tiny
# ============================================================
model_vit = DualViT(num_classes=6).to(device)
model_vit = train_model(
    model_vit, train_loader, val_loader, device,
    save_path="/content/drive/MyDrive/faces/vit_best.pth",
    model_name="Swin-Tiny",
    backbone_lr=5e-6,
    head_lr=5e-5,
)
del model_vit; torch.cuda.empty_cache(); gc.collect()



model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

/tmp/ipykernel_2188/2153278407.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=use_amp)



[Swin-Tiny] FASE 1 — Warmup classifier (5 epoch)
  [Swin-Tiny] Epoch 1/5


/tmp/ipykernel_2188/1462884191.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
/tmp/ipykernel_2188/1462884191.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


    Train Loss: 1.2067 | Val Loss: 0.8557 | Val Acc: 0.8160
    ✓ Saved (acc=0.8160)
  [Swin-Tiny] Epoch 2/5
    Train Loss: 0.9517 | Val Loss: 0.9034 | Val Acc: 0.7847
  [Swin-Tiny] Epoch 3/5
    Train Loss: 0.9023 | Val Loss: 0.8660 | Val Acc: 0.8194
    ✓ Saved (acc=0.8194)
  [Swin-Tiny] Epoch 4/5
    Train Loss: 0.8787 | Val Loss: 0.8024 | Val Acc: 0.8611
    ✓ Saved (acc=0.8611)
  [Swin-Tiny] Epoch 5/5
    Train Loss: 0.8126 | Val Loss: 0.7981 | Val Acc: 0.8681
    ✓ Saved (acc=0.8681)

[Swin-Tiny] FASE 2 — Fine-tune semua layer (25 epoch)
  [Swin-Tiny] Epoch 6/30
    Train Loss: 0.7878 | Val Loss: 0.7974 | Val Acc: 0.8889
    ✓ Saved (acc=0.8889)
  [Swin-Tiny] Epoch 7/30
    Train Loss: 0.7793 | Val Loss: 0.7721 | Val Acc: 0.8924
    ✓ Saved (acc=0.8924)
  [Swin-Tiny] Epoch 8/30
    Train Loss: 0.7411 | Val Loss: 0.7714 | Val Acc: 0.8993
    ✓ Saved (acc=0.8993)
  [Swin-Tiny] Epoch 9/30
    Train Loss: 0.7267 | Val Loss: 0.7772 | Val Acc: 0.8924
  [Swin-Tiny] Epoch 10/30
    Trai

9

In [22]:

# ============================================================
# CELL 22 — Load best weights untuk ensemble
# ============================================================
model1 = DualEfficientNet(num_classes=6)
model1.load_state_dict(torch.load(
    "/content/drive/MyDrive/faces/eff_best.pth", map_location=device))
model1.to(device).eval()

model2 = DualConvNeXt(num_classes=6)
model2.load_state_dict(torch.load(
    "/content/drive/MyDrive/faces/cnx_best.pth", map_location=device))
model2.to(device).eval()

model3 = DualViT(num_classes=6)
model3.load_state_dict(torch.load(
    "/content/drive/MyDrive/faces/vit_best.pth", map_location=device))
model3.to(device).eval()

print("✓ Semua model loaded untuk ensemble")



✓ Semua model loaded untuk ensemble


In [23]:

# ============================================================
# CELL 23 — Inference + Ensemble (soft voting)
# ============================================================
preds_list = []
ids_list   = []

with torch.no_grad():
    for img_tight, img_loose, img_ids in tqdm(test_loader, desc="Inference"):
        img_tight = img_tight.to(device)
        img_loose = img_loose.to(device)

        p1   = F.softmax(model1(img_tight, img_loose), dim=1)
        p2   = F.softmax(model2(img_tight, img_loose), dim=1)
        p3   = F.softmax(model3(img_tight, img_loose), dim=1)
        prob = (p1 + p2 + p3) / 3
        pred = prob.argmax(1).cpu().numpy()

        preds_list.extend(pred.tolist())
        ids_list.extend(list(img_ids))

print(f"✓ Inference selesai: {len(preds_list)} prediksi")



Inference: 100%|██████████| 51/51 [03:55<00:00,  4.62s/it]

✓ Inference selesai: 404 prediksi


In [24]:

# ============================================================
# CELL 24 — Buat submission CSV
# ============================================================
id_to_label = {v: k for k, v in label_map.items()}
pred_labels = [id_to_label[p] for p in preds_list]

submission = pd.DataFrame({"id": ids_list, "label": pred_labels})
submission = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("/content/submission_ensemble.csv", index=False)

print(submission.head(10))
print(f"\nShape : {submission.shape}")
print(f"Labels: {submission['label'].value_counts().to_dict()}")
print("✓ Submission tersimpan di /content/submission_ensemble.csv")


         id           label
0  test_001     fake_screen
1  test_002  fake_mannequin
2  test_003       fake_mask
3  test_004      realperson
4  test_005    fake_printed
5  test_006  fake_mannequin
6  test_007       fake_mask
7  test_008  fake_mannequin
8  test_009       fake_mask
9  test_010     fake_screen

Shape : (404, 2)
Labels: {'realperson': 89, 'fake_mask': 81, 'fake_screen': 71, 'fake_printed': 60, 'fake_mannequin': 55, 'fake_unknown': 48}
✓ Submission tersimpan di /content/submission_ensemble.csv
